In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [2]:
import os, shutil

aligned_root = "utkface_aligned_cropped"
target_dir = "aligned_dataset"
os.makedirs(target_dir, exist_ok=True)

for subfolder in ["crop_part1", "UTKFace"]:
    folder = os.path.join(aligned_root, subfolder)
    for img in os.listdir(folder):
        src = os.path.join(folder, img)
        dst = os.path.join(target_dir, img)
        shutil.copy(src, dst)

print("✅ Combined aligned dataset into 'aligned_dataset/'")

✅ Combined aligned dataset into 'aligned_dataset/'


In [3]:
import os
import cv2
import numpy as np

dataset_dir = "aligned_dataset"
image_files = os.listdir(dataset_dir)

# Check a few random files
print("Sample filenames:", image_files[:5])

# Extract age labels and check some sample images
for img_name in image_files[:5]:
    try:
        age = int(img_name.split("_")[0])
        img_path = os.path.join(dataset_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Failed to load: {img_name}")
        else:
            print(f"✅ {img_name} | Age: {age} | Shape: {img.shape}")
    except Exception as e:
        print(f"❌ Error processing {img_name}: {e}")

Sample filenames: ['100_0_0_20170112213500903.jpg.chip.jpg', '100_0_0_20170112215240346.jpg.chip.jpg', '100_1_0_20170110183726390.jpg.chip.jpg', '100_1_0_20170112213001988.jpg.chip.jpg', '100_1_0_20170112213303693.jpg.chip.jpg']
✅ 100_0_0_20170112213500903.jpg.chip.jpg | Age: 100 | Shape: (200, 200, 3)
✅ 100_0_0_20170112215240346.jpg.chip.jpg | Age: 100 | Shape: (200, 200, 3)
✅ 100_1_0_20170110183726390.jpg.chip.jpg | Age: 100 | Shape: (200, 200, 3)
✅ 100_1_0_20170112213001988.jpg.chip.jpg | Age: 100 | Shape: (200, 200, 3)
✅ 100_1_0_20170112213303693.jpg.chip.jpg | Age: 100 | Shape: (200, 200, 3)


In [4]:
image_paths = []
age_labels = []

for img_name in image_files:
    try:
        age = int(img_name.split("_")[0])
        img_path = os.path.join(dataset_dir, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            image_paths.append(img_path)
            age_labels.append(age)
    except Exception as e:
        print(f"⚠️ Skipped {img_name}: {e}")

print(f"✅ Total valid images: {len(image_paths)}")

✅ Total valid images: 23709


In [5]:
from sklearn.model_selection import train_test_split

# Split 80% train, 20% test
train_paths, test_paths, train_ages, test_ages = train_test_split(
    image_paths, age_labels,
    test_size=0.2,
    random_state=42
)

print(f"✅ Training samples: {len(train_paths)}")
print(f"✅ Testing samples: {len(test_paths)}")

✅ Training samples: 18967
✅ Testing samples: 4742


In [6]:
IMG_SIZE = 64

In [7]:
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0  # normalize to [0, 1]
    return image, label

In [8]:
# Define age bins (10 bins: 0-10, 11-20, ..., 91+)
bin_edges = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 120]  # 10 bins

def age_to_bin(age):
    for i in range(len(bin_edges)-1):
        if bin_edges[i] <= age <= bin_edges[i+1]:
            return i
    return len(bin_edges) - 2

train_bins = [age_to_bin(age) for age in train_ages]
test_bins = [age_to_bin(age) for age in test_ages]

bin_labels = [
    "0-10", "11-20", "21-30", "31-40", "41-50",
    "51-60", "61-70", "71-80", "81-90", "91+"
]

print(f"✅ Converted ages to {len(bin_labels)} bins")

✅ Converted ages to 10 bins


In [9]:
# Recreate train dataset with bin labels
train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_bins))
train_dataset = train_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

# Recreate test dataset with bin labels
test_dataset = tf.data.Dataset.from_tensor_slices((test_paths, test_bins))
test_dataset = test_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

print("✅ Rebuilt tf.data datasets with binned labels")

✅ Rebuilt tf.data datasets with binned labels


In [10]:
def assign_bin(age):
    if age <= 20:
        return 0  # 0-20
    elif age <= 40:
        return 1  # 21-40
    elif age <= 60:
        return 2  # 41-60
    else:
        return 3  # 61+

In [11]:
# Assuming you already have train_ages and test_ages as numpy arrays
train_bins = np.array([assign_bin(age) for age in train_ages])
test_bins = np.array([assign_bin(age) for age in test_ages])

In [12]:
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_bins))
train_dataset = train_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((test_paths, test_bins))
test_dataset = test_dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

print("✅ Rebuilt tf.data datasets with 4-class age bins")

✅ Rebuilt tf.data datasets with 4-class age bins


In [13]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')  # now 4 bins
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

C:\Users\HP\anaconda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 683,716 (2.61 MB)

 Trainable params: 683,716 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    "age_model_best_4bins.h5", monitor="val_accuracy", verbose=1,
    save_best_only=True, mode="max"
)
early_stop = EarlyStopping(
    monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1
)

history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8413 - loss: 0.3958
Epoch 1: val_accuracy improved from -inf to 0.77246, saving model to age_model_best_4bins.h5


593/593 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.8413 - loss: 0.3958 - val_accuracy: 0.7725 - val_loss: 0.6455
Epoch 2/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8410 - loss: 0.3839
Epoch 2: val_accuracy did not improve from 0.77246
593/593 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.8410 - loss: 0.3839 - val_accuracy: 0.7632 - val_loss: 0.6658
Epoch 3/10
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8501 - loss: 0.3609
Epoch 3: val_accuracy did not improve from 0.77246
593/593 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.8501 - loss: 0.3610 - val_accuracy: 0.7579 - val_loss: 0.6874
Epoch 4/10
592/593 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8603 - loss: 0.3380
Epoch 4: val_accuracy did not improve from 0.77246
593/593 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.8603 - loss: 0.3380 - val_accuracy: 0.7598 - val_loss: 0.7974
Epoch 5/10
593/593 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8712 - loss: 0.3179
Epoch 5: val_accuracy d

In [17]:
loss, acc = model.evaluate(test_dataset)
print(f"✅ Final Test Accuracy: {acc * 100:.2f}%")

149/149 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.7821 - loss: 0.6429
✅ Final Test Accuracy: 77.25%


In [18]:
model.save("final_age_model_4bins.h5")